In [24]:
import sys
import os

sys.path.append(os.path.abspath("../src"))

In [25]:
from hypothesis_tests import chi_square_test, t_test

In [26]:
import pandas as pd

df = pd.read_csv("../data/insurance_data.csv")

# Create Margin
df["Margin"] = df["TotalPremium"] - df["TotalClaims"]

# Create Claim Flag (Frequency)
df["HasClaim"] = (df["TotalClaims"] > 0).astype(int)

In [27]:
# Claim Frequency
def claim_frequency(group):
    return group["HasClaim"].mean()

# Claim Severity
def claim_severity(group):
    return group.loc[group["TotalClaims"] > 0, "TotalClaims"].mean()

# Margin
def margin(group):
    return group["Margin"].mean()

In [28]:
from scipy.stats import chi2_contingency

In [29]:
from scipy.stats import ttest_ind

HYPOTHESIS 1: PROVINCES

In [30]:
province_table = pd.crosstab(df["Province"], df["HasClaim"])

p1 = chi_square_test(province_table)

print(p1)

0.07608907812609692


HYPOTHESIS 2: ZIP CODES (Risk differences)

In [31]:
zip_table = pd.crosstab(df["ZipCode"], df["HasClaim"])

p2 = chi_square_test(zip_table)

print(p2)

0.015451174994488507


HYPOTHESIS 3: ZIP CODES (Margin difference)

In [32]:
top_zips = df["ZipCode"].value_counts().head(2).index

a = df[df["ZipCode"] == top_zips[0]]
b = df[df["ZipCode"] == top_zips[1]]

p3 = t_test(a, b, "Margin")

print(p3)

0.26423431242313866


HYPOTHESIS 4: GENDER (Risk difference)

In [33]:
gender_table = pd.crosstab(df["Gender"], df["HasClaim"])

p4 = chi_square_test(gender_table)

print(p4)

0.9638306173980254


In [34]:
results = pd.DataFrame([
    ["Province", "Chi-square", p1, "Reject" if p1 < 0.05 else "Fail to Reject"],
    ["ZipCode", "Chi-square", p2, "Reject" if p2 < 0.05 else "Fail to Reject"],
    ["ZipCode Margin", "T-test", p3, "Reject" if p3 < 0.05 else "Fail to Reject"],
    ["Gender", "Chi-square", p4, "Reject" if p4 < 0.05 else "Fail to Reject"],
], columns=["Hypothesis", "Test", "P-value", "Decision"])

results

,Hypothesis,Test,P-value,Decision
0,Province,Chi-square,0.076089,Fail to Reject
1,ZipCode,Chi-square,0.015451,Reject
2,ZipCode Margin,T-test,0.264234,Fail to Reject
3,Gender,Chi-square,0.963831,Fail to Reject


If province is significant:

We reject H₀ for provinces (p < 0.05).
This indicates that risk differs significantly across provinces.
Pricing should be adjusted regionally to reflect higher-risk areas.